J5 — Valider sans se mentir

**Séance 5.** Un score n’existe pas tout seul : il existe sous un découpage nommé. Changer le
découpage change le score sans changer le modèle. Aujourd’hui, on apprend à choisir le découpage
avant de mesurer, et à reconnaître les trois façons de se mentir.

> **La règle de la séance :** le découpage se **décide**, il ne se subit pas. Un découpage qui
> place le même patient des deux côtés rend un chiffre qui ne veut rien dire — et il ne prévient
> pas.

**Ce que la séance produit :** trois découpages du même modèle, mesurés côte à côte. La fuite de
prétraitement, introduite puis réparée. Le tableau des plis. Et les deux périodes de la table.

**Les trois façons de se mentir, que la séance démonte :** placer le même patient des deux côtés.
Apprendre une statistique avant le découpage. Mélanger le passé et l’avenir.

**Où elle s’arrête, et c’est la frontière avec la séance 6 :** ici on **valide**. On apprend à ne
pas se mentir sur un score. La séance 6 **prévoit** : elle produira une prévision et la jugera
contre un naïf.

**La mise en place n’est pas reconstruite ici** : elle est **extraite** de `notebook_J1.ipynb` et
de `notebook_J3.ipynb`, qui restent les sources uniques.


## Mise en place (fournie)

Ces cellules sont **extraites** des carnets précédents. Exécutez-les de haut en bas, sans rien y
modifier. Elles apportent les données, la table de travail, la référence, le découpage, le modèle
et la **barre de bruit** posée en séance 3.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5).
Le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("les cinq premières lignes \n")
display(train.head())


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("la table de travail \n")
print("colonnes conservées :", X.shape[1])
print("colonnes écartées (cardinalité trop forte) :", TROP_FIN)
print("colonnes catégorielles :", CAT)


### La cible, et la référence


In [ ]:
print("la distribution de la cible \n")
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])
display(repartition)

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())
print(f"la référence (répondre toujours la classe majoritaire) : {reference:.2f} %")


In [ ]:
# `y` entre ici ; la référence a déjà été calculée avec la distribution ci-dessus —
y = train[CIBLE]


### Le découpage, le modèle, et les baselines


In [ ]:
# les baselines, sur le même jeu de validation — et la barre qui décide
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 1) la réponse constante : le concurrent le plus bête, et le plus dur à battre
constante = 100 * max(yval.mean(), 1 - yval.mean())

# 2) la règle du délai : « j'annonce une absence si le délai dépasse N jours ».
#    N se choisit sur les lignes d'apprentissage, pas sur celles de contrôle.
delai_max = int(Xtr['delai_jours'].max())
taux_par_seuil = {s: float(((Xtr['delai_jours'] > s).astype(int) == ytr).mean())
                  for s in (0, 1, 3, 7, 14, 30) if s < delai_max}
seuil_appris = max(taux_par_seuil, key=taux_par_seuil.get)
regle = (Xval['delai_jours'] > seuil_appris).astype(int)

# et la règle qui ne dirait jamais rien ? c'est la réponse constante, au point près
silence = (Xval['delai_jours'] > delai_max).astype(int)

# 3) la famille linéaire : elle a besoin de nombres, on lui en prépare — la recette des
#    séances précédentes (newton-cholesky : le solveur qui converge sur cette table)
X_n = pd.get_dummies(X, columns=CAT).fillna(0)   # 0 = trou comblé ; le linéaire ne lit pas les NaN
Xtr_n, Xval_n = X_n.loc[Xtr.index], X_n.loc[Xval.index]
lineaire = LogisticRegression(solver='newton-cholesky').fit(Xtr_n, ytr)

# 4) le modèle du cours, celui des séances précédentes
modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

# la barre : l'erreur-type de l'exactitude sur CE jeu de validation
barre = 100 * ((constante / 100) * (1 - constante / 100) / len(yval)) ** 0.5

baselines = pd.DataFrame([
    {'baseline': 'toujours présent',
     'exactitude (%)': round(constante, 2)},
    {'baseline': f'règle du délai — absent si le délai dépasse {seuil_appris} j',
     'exactitude (%)': round(100 * (regle == yval).mean(), 2)},
    {'baseline': 'absent si ce patient a déjà manqué un rendez-vous',
     'exactitude (%)': round(100 * ((Xval['taux_absence_passe'] > 0).astype(int) == yval).mean(), 2)},
    {'baseline': 'la famille linéaire',
     'exactitude (%)': round(100 * lineaire.score(Xval_n, yval), 2)},
    {'baseline': 'le modèle du cours',
     'exactitude (%)': round(100 * modele.score(Xval, yval), 2)},
])
display(baselines)

ecart = 100 * modele.score(Xval, yval) - constante
print(f'→ le modèle dépasse la réponse constante de {ecart:+.2f} point,')
print(f'  soit {ecart / barre:.1f} erreurs-types de ce jeu de validation ({barre:.2f} point).')
print(f'  la règle du délai, elle, fait {100 * (regle == yval).mean():.2f} % : elle alerte sur')
print(f'  {int(regle.sum())} lignes et perd contre la réponse constante.')
print(f'  et la règle qui ne dirait jamais rien ferait {100 * (silence == yval).mean():.2f} % —')
print('  exactement la réponse constante. Une règle en place se mesure avant d’être opposée')
print('  à un modèle ; ce que cela dit d’une exactitude reste ouvert.')


### Le rituel : annoncer avant de mesurer


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))


## 1. Le découpage qui ment

Un patient revient. Sur cette table, **32,63 % des patients ont plus d’un rendez-vous**, et ces
patients portent **55,11 % des lignes**. Le plus assidu en a **161**. Le modèle, lui, ne voit
jamais l’identifiant du patient : la colonne est écartée de la table de travail.

Il n’a donc pas besoin de l’identifiant pour reconnaître un patient. Il lui suffit de retrouver,
dans le jeu de contrôle, des lignes qui ressemblent à celles qu’il a déjà apprises. Un découpage
au hasard place **le même patient des deux côtés**. Le score se calcule alors sur des lignes
déjà vues — et aucun nouveau patient ne sera jamais dans ce cas.

Ce qu’on regarde : le même modèle, trois découpages, et le nombre de patients présents des deux
côtés. L’écart de score est la mesure ; le nombre de patients partagés est le mécanisme.

Ce que cette section **ne tranche pas** : le découpage de votre sujet. Il dépend de la structure de
vos données, et il se décide avant de mesurer.


In [ ]:
# les trois découpages, sur la même table — le même patient peut se trouver des deux côtés
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, train_test_split

pos = np.arange(len(X))
al_app, al_cont = train_test_split(pos, test_size=0.2, random_state=0)
st_app, st_cont = train_test_split(pos, test_size=0.2, stratify=y, random_state=0)
gp_app, gp_cont = next(iter(GroupKFold(n_splits=5).split(X, y, groups=train['patient_id'])))

fig, ax = plt.subplots(figsize=(9, 3))
for k, (nom, app, cont) in enumerate([('au hasard', al_app, al_cont),
                                      ('stratifié', st_app, st_cont),
                                      ('par patient', gp_app, gp_cont)]):
    ax.scatter(app, [k] * len(app), s=1, color='#b9b9b9')
    ax.scatter(cont, [k] * len(cont), s=1.5, color='#c0392b')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['au hasard', 'stratifié', 'par patient'])
ax.set_xlabel('les 149 894 lignes, dans l’ordre de la table')
ax.set_title('gris : apprentissage · rouge : contrôle')
plt.show()

partages = len(set(train['patient_id'].iloc[al_app]).intersection(train['patient_id'].iloc[al_cont]))
print(f'patients présents des deux côtés, au hasard : {partages} sur {len(al_cont)} lignes de contrôle')


In [ ]:
# le même modèle, trois découpages — et l'écart que personne ne regarde
from sklearn.ensemble import HistGradientBoostingClassifier

def score_du_decoupage(app, cont):
    """Le modèle du cours, entraîné sur un côté et jugé sur l'autre."""
    m = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[app], y.iloc[app])
    return 100 * m.score(X.iloc[cont], y.iloc[cont])

prevoir('un découpage par patient', 'le score baisse',
        'les lignes de contrôle ne ressemblent plus à des lignes déjà apprises')
scores = {'au hasard': score_du_decoupage(al_app, al_cont),
          'stratifié (le découpage du cours)': score_du_decoupage(st_app, st_cont),
          'par patient': score_du_decoupage(gp_app, gp_cont)}
noter('un découpage par patient', 'GroupKFold sur patient_id',
      'les lignes de contrôle ne ressemblent plus à des lignes déjà apprises',
      scores['par patient'])

display(pd.DataFrame({'découpage': list(scores), 'exactitude (%)': [round(v, 2) for v in scores.values()]}))
ecart = scores['stratifié (le découpage du cours)'] - scores['par patient']
print(f'→ l’écart entre le découpage du cours et le découpage par patient : {ecart:+.2f} point.')
print(f'  la barre de bruit vaut {barre:.2f} : cet écart-là ne se lit pas comme un gain.')
print(f'  et {100 * partages / len(al_cont):.1f} % des lignes de contrôle portaient un patient')
print('  déjà vu à l’apprentissage. Le score ne le dit pas — le découpage, si.')


> **Question →** `Devoir_maison_3.ipynb`, **question 1 (carnet J5)** — l’écart entre les trois
> découpages sur vos données, et lequel ment.


**Une chose que la mesure montre, et qui surprend :** l’écart de score est **petit**. Ici, il vaut
**0,09 point**, moins que la barre de bruit. Un découpage au hasard ne gonfle donc pas le score de
dix points : il le gonfle d’un rien, et c’est plus grave. Un chiffre franchement faux se repère ; un
chiffre faux d’un dixième de point se garde, se publie, et se paie en production.


### La démonstration, découpage par découpage

Une même paire de courbes — le score d’**apprentissage** et le score de **contrôle** — sous les
trois découpages. C’est le geste des séances 3 et 4, appliqué ici au découpage lui-même.

**La barre de bruit, écrite deux fois.** En symboles : $\sqrt{p(1-p)/n}$, et la barre vaut **deux**
fois cet écart-type, soit environ 95 % des tirages. En mots : l’erreur-type d’une exactitude sur
$n$ lignes de contrôle vaut **0,22 point** sur 29 979 lignes, **1,22 point** sur 1 000.

Ce qu’on regarde : les deux courbes, dans les trois découpages.


In [ ]:
# la même paire de courbes sous les trois découpages : l'apprentissage, et le contrôle
from sklearn.ensemble import HistGradientBoostingClassifier
import matplotlib.pyplot as plt

plan = [('au hasard', al_app, al_cont), ('stratifié', st_app, st_cont), ('par patient', gp_app, gp_cont)]
fig, (gauche, droite) = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for nom, app_i, cont_i in plan:
    apprentissage, controle = [], []
    for arbres in (5, 10, 20, 30):
        m = HistGradientBoostingClassifier(max_iter=arbres, early_stopping=False, random_state=0).fit(X.iloc[app_i], y.iloc[app_i])
        apprentissage.append(100 * m.score(X.iloc[app_i], y.iloc[app_i]))
        controle.append(100 * m.score(X.iloc[cont_i], y.iloc[cont_i]))
    gauche.plot((5, 10, 20, 30), apprentissage, marker='o', label=nom)
    droite.plot((5, 10, 20, 30), controle, marker='o', label=nom)
    print(f'  {nom:<12} apprentissage {apprentissage[-1]:5.2f} % | contrôle {controle[-1]:5.2f} % (maximum {max(controle):5.2f} %)')
gauche.set_title('le score d’apprentissage'); gauche.set_xlabel('arbres'); gauche.set_ylabel('exactitude (%)')
droite.set_title('le score de contrôle'); droite.set_xlabel('arbres'); droite.legend(fontsize=8)
plt.show()
print('→ l’apprentissage monte pareil dans les trois cas : il ne dit rien du découpage.')
print('  Le contrôle, lui, plafonne plus bas et plus tôt sous le découpage par patient —')
print('  et c’est la seule des deux courbes qui parle du monde réel.')


## 2. La fuite de prétraitement

La séance 2 a montré les colonnes qui portent la réponse : la durée de la consultation, le motif
d’absence. Les écarter a suffi. Il reste une fuite qui n’a l’air de rien, parce qu’elle ne touche
à aucune colonne : elle tient à **l’ordre des opérations**.

Un prétraitement apprend toujours quelque chose de la table. Une moyenne, une liste de catégories,
un taux. Si cet apprentissage est fait **avant** le découpage, il a vu les lignes de contrôle. Le
modèle reçoit alors un renseignement qui vient, en partie, de ce qu’il doit prédire.

Cette section emploie un renseignement qui a le droit d’exister : **le taux d’absence passé du
patient**. Le cours s’en sert déjà, et il est dans la table sous le nom `taux_absence_passe`. Une
colonne qui parle du même patient est exactement ce qu’un découpage par groupes protège.

Ce qu’on regarde : la même colonne, calculée de trois façons. Sur **toute** la table, étiquette
comprise : c’est la fuite. Sur l’apprentissage seul, en laissant chaque ligne contribuer à sa
propre valeur. Puis **hors pli**, c’est-à-dire sans qu’aucune ligne serve à calculer sa propre
valeur. Trois chiffres, et un seul se défend.

Ce que cette section **ne tranche pas** : le lissage d’un tel encodage, et la sélection de modèle
à l’intérieur d’un pli. Les deux sont nommés, aucun des deux n’est développé.


In [ ]:
# 1. la fuite : la colonne est calculée sur toute la table, étiquette comprise
def historique(appris, lignes):
    """Le taux d’absence passé du patient : appris sur `appris`, appliqué à `lignes`.
       Un patient inconnu de `appris` reçoit le taux moyen de `appris`."""
    taux = appris.groupby('patient_id')[CIBLE].mean()
    return lignes['patient_id'].map(taux).fillna(appris[CIBLE].mean()).to_numpy()

prevoir('l’historique du patient', 'calculé sur toute la table, il fait exploser le score',
        'chaque ligne de contrôle a contribué à sa propre valeur')
X_fuite = X.copy()
X_fuite['historique_patient'] = historique(train, train)
m_fuite = HistGradientBoostingClassifier(random_state=0).fit(X_fuite.iloc[st_app], y.iloc[st_app])
score_fuite = 100 * m_fuite.score(X_fuite.iloc[st_cont], y.iloc[st_cont])
noter('l’historique du patient', 'calculé sur toute la table',
      'chaque ligne de contrôle a contribué à sa propre valeur', score_fuite)
print(f'  la fuite, sur toute la table : {score_fuite:.2f} %')


In [ ]:
# 2. le geste naturel : apprendre la colonne sur l'apprentissage, et l'y laisser telle quelle
app, cont = train.iloc[st_app], train.iloc[st_cont]
Xa = X.iloc[st_app].copy()
Xa['historique_patient'] = historique(app, app)
m_naif = HistGradientBoostingClassifier(random_state=0).fit(Xa, y.iloc[st_app])
score_naif = 100 * m_naif.score(Xa, y.iloc[st_app])
score_naif_controle = 100 * m_naif.score(X_fuite.iloc[st_cont].assign(
    historique_patient=historique(app, cont)), y.iloc[st_cont])
print(f'  l’apprentissage se voit lui-même : {score_naif:.2f} % sur lui-même, {score_naif_controle:.2f} % au contrôle')
print('→ la colonne est parfaite là où elle a été apprise, et trompeuse ailleurs : le modèle')
print('  s’y appuie, puis se casse sur des patients qu’il n’a jamais vus. Ce chiffre est')
print(f'  pire que celui du cours sans la colonne ({scores["stratifié (le découpage du cours)"]:.2f} %).')


In [ ]:
# 3. la réparation : hors pli sur l'apprentissage, et les plis ne voient jamais leur propre ligne
from sklearn.model_selection import StratifiedKFold

taux = app.groupby('patient_id')[CIBLE].mean()
Xa['historique_patient'] = np.nan
for a, b in StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(Xa, y.iloc[st_app]):
    par_pli = app.iloc[a].groupby('patient_id')[CIBLE].mean()
    Xa.iloc[b, Xa.columns.get_loc('historique_patient')] = (
        app.iloc[b]['patient_id'].map(par_pli).fillna(app[CIBLE].mean()).to_numpy())
Xa['historique_patient'] = Xa['historique_patient'].fillna(app[CIBLE].mean())

Xl = X.iloc[st_cont].copy()
Xl['historique_patient'] = cont['patient_id'].map(taux).fillna(app[CIBLE].mean()).to_numpy()
m_propre = HistGradientBoostingClassifier(random_state=0).fit(Xa, y.iloc[st_app])
propre = 100 * m_propre.score(Xl, y.iloc[st_cont])

inconnus = cont['patient_id'].isin(app['patient_id']).eq(False).mean()
print(f'  hors pli sur l’apprentissage : {propre:.2f} %')
print(f'→ les trois états du même renseignement : {score_fuite:.2f} % · {score_naif_controle:.2f} % · {propre:.2f} %.')
print(f'  Le gain honnête vaut {propre - scores["stratifié (le découpage du cours)"]:+.2f} point — à peine')
print(f'  plus que la barre ({barre:.2f}). Il n’est pas démontré, et il se dit tel quel.')
print(f'  Et {100 * inconnus:.1f} % des lignes de contrôle portent un patient inconnu : en production,')
print('  ce sont elles qui décident, et elles n’ont pas d’historique.')


**Et l’avance du modèle tient à cette seule colonne.** Mesuré : sans `taux_absence_passe`, le même
modèle retombe à **81,76 %**, c’est-à-dire la réponse constante (**81,75 %**). Tout ce que le modèle
sait de plus que « tout le monde vient », il le tient d’une colonne qui parle du patient.

> **Question →** `Devoir_maison_3.ipynb`, **question 2 (carnet J5)** — introduire la fuite, mesurer
> ce qu’elle rapporte, puis la corriger dans une chaîne.


### La fuite, lue au niveau de la perte

L’exactitude raconte une partie de l’histoire. La **perte** — la log-loss, celle que l’entraînement
minimise — la raconte en entier, et les trois états de la même colonne y ont **trois signatures**
différentes.

**La perte, écrite deux fois.** En symboles : $-\log(p)$ pour la vraie classe, moyenne sur les
lignes. En mots : la punition d’une annonce. Elle est d’autant plus forte que l’annonce était
confiante et fausse. Annoncer 0,01 à un patient absent coûte **4,6** ; annoncer 0,99 coûte **0,01**.

Ce qu’on regarde : la perte d’apprentissage et celle de contrôle, pour les trois versions.


In [ ]:
# les trois états de la colonne, lus au niveau de la perte
from sklearn.metrics import log_loss

X_naif = X.iloc[st_app].copy()
X_naif['historique_patient'] = historique(app, app)

def pertes(modele, X_app, X_cont, nom):
    """La perte d’apprentissage et la perte de contrôle du même modèle."""
    app_perte = log_loss(y.iloc[st_app], modele.predict_proba(X_app)[:, 1])
    cont_perte = log_loss(y.iloc[st_cont], modele.predict_proba(X_cont)[:, 1])
    print(f'  {nom:<28} apprentissage {app_perte:6.4f} | contrôle {cont_perte:6.4f}')

pertes(m_fuite, X_fuite.iloc[st_app], X_fuite.iloc[st_cont], 'la fuite (toute la table)')
pertes(m_naif, X_naif, Xl, 'le geste naïf')
pertes(m_propre, Xa, Xl, 'hors pli (honnête)')
print('→ trois signatures : la fuite écrase les deux pertes, le geste naïf fait exploser l’écart,')
print('  et la version hors pli laisse un écart petit. Une perte qui se creuse est le signe le')
print('  plus sûr ; une exactitude qui se creuse, seulement le plus visible.')


## 3. Les variantes de découpage, et ce que chacune protège

Un découpage est une hypothèse sur le monde. Il dit ce qui, dans vos données, se répète et ne doit
donc pas servir deux fois. Trois familles suffisent à couvrir la table du cours. Et répéter un
découpage donne plusieurs scores : on les rend alors en moyenne **et** en dispersion :

| Variante | Ce qu’elle protège |
|---|---|
| k-fold | contre un découpage chanceux : cinq découpages, cinq scores |
| stratifié | contre un pli sans positifs, quand la classe est rare |
| par groupes | contre l’entité qui revient : le patient, la clinique, le magasin |
| temporel | contre le futur qui enseigne le passé |
| leave-one-out | rien de plus ici : autant d’évaluations que de lignes, et chaque score voit presque toute la table |

Ce qu’on regarde : cinq plis stratifiés et cinq plis par groupes, sur le même modèle. La moyenne
et l’écart-type disent si l’écart entre les deux découpages dépasse la dispersion des plis.

**La dispersion, écrite deux fois.** En symboles : la moyenne des scores de pli, $\bar{s}$, et leur
écart-type, $\sigma$. En mots : ce que le chiffre vaudrait si l’on avait tiré d’**autres** plis —
donc ce qu’un seul tirage ne peut pas montrer.

Ce que cette section **ne tranche pas** : la comparaison appariée de deux modèles sur un même
découpage, et l’imbrication d’une sélection de modèle dans un pli. Les deux reviendront plus tard.


### La démonstration, pli par pli

Cinq plis de chaque façon, dessinés puis chiffrés : le même modèle, cinq fois, sous deux
découpages. Ce qu’on lit dans le tableau, c’est la **dispersion** — pas la moyenne.


In [ ]:
# cinq plis de chaque façon : la moyenne, et la dispersion autour d'elle
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, StratifiedKFold

plis_stratifie = list(StratifiedKFold(n_splits=5).split(X, y))
plis_groupes = list(GroupKFold(n_splits=5).split(X, y, groups=train['patient_id']))

fig, ax = plt.subplots(figsize=(9, 2.6))
for k, (nom, plis) in enumerate([('stratifié', plis_stratifie), ('par patient', plis_groupes)]):
    for j, (_, cont) in enumerate(plis):
        ax.scatter(cont, [k] * len(cont), s=1, color=f'C{j}')
ax.set_yticks([0, 1])
ax.set_yticklabels(['stratifié', 'par patient'])
ax.set_xlabel('les 149 894 lignes, dans l’ordre de la table — une couleur par pli')
ax.set_title('chaque couleur est un jeu de contrôle')
plt.show()


In [ ]:
# et les deux mêmes découpages, chiffrés
def scores_des_plis(plis):
    """Un score par pli, dans l’ordre des plis."""
    out = []
    for app, cont in plis:
        m = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[app], y.iloc[app])
        out.append(100 * m.score(X.iloc[cont], y.iloc[cont]))
    return out

stratifies = scores_des_plis(plis_stratifie)
groupes = scores_des_plis(plis_groupes)
tableau_plis = pd.DataFrame({
    'découpage': ['stratifié', 'par patient'],
    'moyenne (%)': [round(np.mean(stratifies), 2), round(np.mean(groupes), 2)],
    'écart-type': [round(np.std(stratifies), 3), round(np.std(groupes), 3)],
    'étendue': [round(max(stratifies) - min(stratifies), 2), round(max(groupes) - min(groupes), 2)],
})
display(pd.DataFrame({'pli': [f'pli {i + 1}' for i in range(5)],
                      'stratifié (%)': [round(v, 2) for v in stratifies],
                      'par patient (%)': [round(v, 2) for v in groupes]}))
display(tableau_plis.set_index('découpage'))
print(f'→ la moyenne ne bouge pas ({np.mean(stratifies) - np.mean(groupes):+.2f} point),')
print(f'  mais l’écart-type augmente : {np.std(stratifies):.3f} contre {np.std(groupes):.3f}.')
print('  Le découpage honnête rend un chiffre moins stable — et c’est cette instabilité')
print('  qui dit la vérité : vos plis ne se ressemblent plus.')


> **Question →** `Devoir_maison_3.ipynb`, **question 3 (carnet J5)** — une validation croisée
> stratifiée sur trois modèles, rendue en moyenne **±** écart-type.


## 4. La validation temporelle

Quand une colonne dit *quand*, le hasard devient faux. Un découpage au hasard laisse le modèle
apprendre sur des lignes de décembre et juger sur des lignes de janvier. Il aura vu la suite avant
de rendre sa copie.

Les deux périodes de ce cours ne se chevauchent pas. L’historique s’arrête au **2025-11-18**. La
table de test, elle, porte des rendez-vous de **2026** : mesuré, **63 205 lignes sur 63 721**, soit
**99,2 %**. Les 516 lignes restantes portent des dates de 2023 à 2025. La séparation est nette,
pas parfaite.

Deux gestes suffisent, et voici leurs noms. `TimeSeriesSplit` découpe l’historique en plis où
l’apprentissage précède toujours le contrôle. Le **backtest à origine glissante** fait la même
chose avec un horizon fixe : on avance l’origine, et on rejoue la mesure.

Ce qu’on regarde : le même modèle, jugé sur les 20 % de lignes les plus récentes, contre son score
au hasard. Le contrôle par patient reste d’actualité : les patients reviennent **au fil du temps**,
mesuré à **45,1 %** des lignes de contrôle même dans cette coupe.

Ce que cette section **ne tranche pas** : la prévision elle-même. Produire un chiffre pour une date
qui n’existe pas encore, et le juger contre un naïf, c’est la séance 6.


In [ ]:
# la ligne du temps, les plis qui avancent — et la table qu'il faut TRIER d'abord
from sklearn.model_selection import TimeSeriesSplit

# `TimeSeriesSplit` suit l'ORDRE DES LIGNES, pas les dates. Cette table est mélangée :
# sans tri préalable, les plis ne sont pas des périodes.
ordre_dates = np.argsort(pd.to_datetime(train['date_rdv']).to_numpy(), kind='stable')
X_trie, y_trie = X.iloc[ordre_dates], y.iloc[ordre_dates]
dates = pd.to_datetime(train['date_rdv']).iloc[ordre_dates]
print('la table triée par date est monotone :', bool(dates.is_monotonic_increasing))

plis_temps_bruts = list(TimeSeriesSplit(n_splits=5).split(X_trie))
fig, ax = plt.subplots(figsize=(9, 2.8))
for i, (app, cont) in enumerate(plis_temps_bruts):
    ax.plot([dates.iloc[app].min(), dates.iloc[app].max()], [i] * 2, lw=6, color='#b9b9b9')
    ax.plot([dates.iloc[cont].min(), dates.iloc[cont].max()], [i] * 2, lw=6, color='#c0392b')
ax.axvline(dates.max(), color='black', ls=':', lw=1)
ax.text(dates.max(), -0.9, ' fin de l’historique', fontsize=8)
ax.set_yticks([])
ax.set_xlabel('les plis de TimeSeriesSplit, sur la table triée par date : gris l’apprentissage, rouge le contrôle')
ax.set_title('le contrôle vient toujours après l’apprentissage — à condition d’avoir trié')
plt.show()
print('l’historique va du', dates.min().date(), 'au', dates.max().date())
print('chaque contrôle est une période qui suit son apprentissage :',
      [str(dates.iloc[cont].min().date()) for _, cont in plis_temps_bruts])


### La démonstration, origine par origine

Cinq origines, et pour chacune le même modèle : ce que la coupe dans le temps coûte, et ce que le
découpage par patient y superpose encore.


In [ ]:
# le même modèle, une coupe dans le temps — et le score honnête
coupe = int(0.8 * len(ordre_dates))
app_t, cont_t = ordre_dates[:coupe], ordre_dates[coupe:]
m_temps = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[app_t], y.iloc[app_t])
score_temps = 100 * m_temps.score(X.iloc[cont_t], y.iloc[cont_t])

# cinq plis, sur la table TRIÉE : c'est la seule façon qu'ils soient des périodes
plis_temps = [100 * HistGradientBoostingClassifier(random_state=0)
              .fit(X_trie.iloc[a], y_trie.iloc[a]).score(X_trie.iloc[b], y_trie.iloc[b])
              for a, b in TimeSeriesSplit(n_splits=5).split(X_trie)]
print(f'  au hasard (le découpage du cours) : {scores["stratifié (le découpage du cours)"]:.2f} %')
print(f'  80 / 20 dans le temps             : {score_temps:.2f} %')
print(f'  TimeSeriesSplit, cinq plis        : {np.mean(plis_temps):.2f} % (étendue {max(plis_temps) - min(plis_temps):.2f})')
print(f'  le détail des cinq plis           : {[round(v, 2) for v in plis_temps]}')
ecart_temps = score_temps - scores['stratifié (le découpage du cours)']
print(f'→ la coupe dans le temps coûte {ecart_temps:+.2f} point, soit {abs(ecart_temps) / barre:.1f} barres de bruit.')
print('  Le modèle n’a pas changé. Ce qui a changé, c’est ce qu’on lui demande :')
print('  prédire la suite, au lieu de retrouver ce qu’il a déjà vu.')
print('  Et l’étendue des cinq plis (0,88) vaut quatre fois celle du stratifié (0,25) : le')
print('  temps ne rend pas seulement le chiffre plus bas, il le rend instable — et c’est')
print('  une information sur le monde, pas un défaut de la mesure.')


> **Question →** `Devoir_maison_3.ipynb`, **question 4 (carnet J5)** — la validation temporelle sur
> les dates de votre sujet, et ce qu’elle coûte.


### Le découpage qui protège des deux

Les patients reviennent **et** le temps avance : sur cette table, aucun découpage simple ne protège
des deux. Le découpage temporel laisse **45,1 %** des lignes de contrôle porter un patient déjà vu
côté apprentissage ; le découpage par patient, lui, mélange les périodes.

Ce qu’on regarde : la version qui protège des deux — on garde la coupe dans le temps, **et** on
retire de l’apprentissage les patients qui apparaissent dans le contrôle. Puis le prix : ces
patients ne servent plus à rien apprendre.

Ce que cette section **ne tranche pas** : ce qu’il faut faire quand retirer ces patients vide une
tranche entière de la table. Le cas se pose sur les petites bases ; il se traite en amont.


In [ ]:
# la version qui protège des DEUX : les patients ne traversent pas, et le temps non plus
patients_controle = set(train['patient_id'].iloc[cont_t])
garde = ~train['patient_id'].iloc[app_t].isin(patients_controle).to_numpy()
app_gt = app_t[garde]
modele_gt = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[app_gt], y.iloc[app_gt])
score_gt = 100 * modele_gt.score(X.iloc[cont_t], y.iloc[cont_t])

print(f'  chronologique simple : {score_temps:.2f} %  ({len(app_t)} lignes d’apprentissage)')
print(f'  groupe ET temporel   : {score_gt:.2f} %  ({len(app_gt)} lignes, {len(app_t) - len(app_gt)} patients retirés)')
partages = set(train['patient_id'].iloc[app_gt]).intersection(patients_controle)
print(f'  patients du contrôle déjà vus : {100 * len(partages) / len(patients_controle):.1f} %')
print(f'  pour mémoire : stratifié {scores["stratifié (le découpage du cours)"]:.2f} % · par patient {scores["par patient"]:.2f} %')
print('→ protéger des deux se paie : le score descend, et c’est ce chiffre-là qui décrit le monde')
print('  réel. Un patient qui revient n’est pas un patient neuf, et une date qui suit n’est pas')
print('  une date au hasard.')


### Et une chose qu’on ne traite pas encore : la prévision

Tout ce carnet juge un modèle sur des lignes **déjà étiquetées**. C’est la bonne façon de valider —
et ce n’est pas encore prévoir.

Prévoir, c’est annoncer un **nombre** pour une date qui n’existe pas encore, et le juger contre un
**naïf** sur un **horizon** nommé. Le geste est différent : l’horizon se met de côté avant de
choisir un modèle, et l’erreur se lit **pas par pas**, pas en moyenne.

→ La séance 6 le fait, sur une autre table du même réseau : la table d’activité.


## 5. Ce qu’on retient, et la note technique

Le projet rend une **note technique de trois pages**, au format imposé. Il tient en quatre
sections, et chacune répond à une question que le lecteur se pose dans cet ordre.

| Section | Ce qu’elle dit | La question du lecteur |
|---|---|---|
| **1. Le problème, et ce que coûte une erreur** | la décision, la métrique choisie, le seuil, et les deux prix | *de quoi parle-t-on, et pourquoi cette mesure ?* |
| **2. Les données** | ce qui entre, ce qui est écarté et pourquoi, ce qui n’existera pas en production | *sur quoi le modèle a-t-il appris ?* |
| **3. Les résultats et les erreurs** | la baseline, le protocole de validation, les chiffres **et leur marge**, les essais échoués, les trente erreurs les plus coûteuses | *le gain est-il réel, et où le modèle se trompe-t-il ?* |
| **4. Les limites** | ce que le modèle ne sait pas faire, ce qui le casserait, ce qu’on vérifierait ensuite | *quand ne faut-il pas s’en servir ?* |

La quatrième section est celle qui se bâcle le plus souvent, et celle qu’on vous reprochera. Une
note qui ne dit pas ses limites n’est pas une note courte : c’est une note incomplète.

> **Question →** `Devoir_maison_3.ipynb`, **question 5 (carnet J5)** — le plan appliqué à votre
> sujet : ce que chaque section contiendra, et la preuve qui l’y autorise.


### La liste de contrôle, à emporter

- **Le découpage se décide avant la mesure.** Il dit ce qui se répète : le patient, la clinique, le
  temps. Le score n’a de sens que sous un découpage nommé.
- **Trois fuites, trois symptômes.** La colonne qui disparaît en production : elle gonfle le score
  franchement. La colonne connue après l’issue : elle le fait exploser. La statistique apprise
  avant le découpage : elle gonfle le score de 13 points quand elle contient l’étiquette de la
  ligne. Apprise naïvement, elle fait **pire que rien**.
- **Un écart se lit contre le bruit.** Sur cinq plis, on rend une moyenne **et** une dispersion. Un
  écart plus petit que la dispersion des plis n’est pas un progrès, c’est un tirage.
- **Le découpage par groupes ne fait pas gagner de points.** Il rend un chiffre qui veut dire
  quelque chose. Ici, la moyenne ne bouge pas (**0,00 point**) et la dispersion double (**0,090**
  contre **0,227**). Ce qui change, c’est la confiance qu’on peut mettre dans le chiffre.
- **Le temps ne se mélange pas.** L’historique s’arrête le **2025-11-18**, le test commence en
  **2026** ; juger dans le temps coûte **0,56 point** — et c’est le chiffre à annoncer.
- **L’avance du modèle tient à une colonne.** Sans `taux_absence_passe`, il retombe à la réponse
  constante : dites-le, plutôt que de laisser croire à un modèle qui comprend la médecine.
